# KoChatGPT 업그레이드 — foundation model 교체로 RLHF 3단계 다시 학습하기

이 노트북에서 내가 한 일은 이렇다.
`airobotlab/KoChatGPT` 의 RLHF 파이프라인(SFT -> RM -> PPO)을 **베이스 모델을 바꿔서 다시 학습**시키고,
바꾸기 전(`skt/kogpt2-base-v2`)과 바꾼 뒤(`skt/ko-gpt-trinity-1.2B-v0.5`)를 정량·정성으로 비교했다.

**개선 축 선택**: 프로젝트가 제시한 세 갈래(기존 데이터셋 정제 / 새 데이터셋 추가 / foundation model 교체) 중
**foundation model 교체**를 골랐다. 이유는 내 실행 환경 때문이다. 로컬 GPU가 통합메모리라 3단계 전체를 돌려도 여유 있었다.  
노드가 경고한 OOM 부담이 거의 없어서, 1.2B 모델로 갈아끼우는 게 괜찮은 선택이었다.

**이 노트북의 순서**

```
0. 실행 환경 (로컬 ROCm 패치)
1. 베이스 모델 확인 — 학습 전에는 무엇을 하는 모델인가
2. 데이터셋 3종 (SFT / RM / PPO)
3. 1단계 SFT   — 대답하는 형식을 가르친다
4. 2단계 RM    — 어느 답이 더 나은지 점수를 매기는 심판을 기른다
5. 3단계 PPO   — 그 점수를 쫓되 예전의 자신에게서 멀어지지 않게 민다
6. 개선 비교분석 — 교체 전후 정성·정량 비교
7. 회고 / 루브릭 자체평가
```

### 읽기 전에 — 이 노트북을 만든 방식 두 가지

**1. 학습 진행바는 텍스트로 옮겨 적었다.**
`tqdm` 진행바는 `ipywidgets` 형식으로 저장되는데 GitHub 은 이걸 렌더링하지 못해서 빨간 오류 줄로만 보인다.
그래서 위젯 출력은 빼고, **노트북 파일에 이미 저장돼 있던 진행바의 최종 상태를 그대로 텍스트로 옮겼다.**
새로 만든 숫자가 아니라 이 파일 안에 기록돼 있던 값이다. RM 의 최종 loss 나 PPO 에피소드별 loss 가
거기 들어 있어서 버릴 수 없었다.

**2. 마지막 "개선 비교분석" 부분은 따로 실행했다.**
비교분석을 하려고 노트북 전체를 다시 돌리면 SFT/RM/PPO 학습이 통째로 다시 돌아가서(3시간 넘는다)
앞의 학습 로그가 전부 덮인다. 그래서 **앞은 그대로 두고 비교분석만 따로 실행해 이어 붙였다.**
그래서 그 지점부터 **셀 실행 번호가 다시 작은 숫자로 시작한다.** 번호를 손으로 고쳐 한 번에 돌린 척하지 않았다.

**미리 밝혀두는 결론 두 가지.** 뒤에서 근거와 함께 다시 적는다.
- 교체로 좋아진 것은 **사실성이 아니라 언어 일관성과 형식 안정성**이다. 두 모델 다 사실오류는 남았다.
- **PPO 단계는 이번 규모에서 이득을 내지 못했다.** 그 이유도 내 실측(RM 점수)으로 설명한다.

## 0. 실행 환경 — 로컬 ROCm 에서 돌리기 위한 패치

노드는 Colab 기준이라 `/content` 경로와 `pip install`, `git clone` 이 들어 있다.
나는 LMS 접속이 안 돼서 로컬에서 돌렸고, 그래서 다음을 바꿨다.

- 경로를 `/content` 에서 내 작업 폴더로 교체
- 설치·clone 셀은 무해화 (aiffel 환경에 이미 설치돼 있다)
- **`requirements.txt` 통설치는 하지 않았다.** 그걸 그대로 돌리면 CUDA 빌드 torch 가
  내 ROCm torch 를 덮어써서 GPU 를 못 쓰게 된다. 필요한 패키지만 개별 설치했다.
- `chatgpt` 패키지는 `colossalai` 의존이 걸려 있어 그 부분을 문자열 패치로 제거하고
  `NaiveStrategy`(단일 GPU)만으로 돌아가게 만들었다.

In [1]:
# [로컬패치] 환경 정리 + chatgpt 패키지 경로 등록 + 산출물 폴더
import os, sys, warnings
# protobuf/TF 노이즈(MessageFactory GetPrototype) 차단 — 무해 경고라 로그만 더럽힘
os.environ['USE_TF']='0'; os.environ['USE_TORCH']='1'
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION']='python'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS']='1'
warnings.filterwarnings('ignore')
sys.path.insert(0, r'/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build')
os.makedirs(r'/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out', exist_ok=True)
print('local paths ready')


local paths ready


In [2]:
# === Colab 호환 셋업 (자동 추가) ===
import torch as _t
_t._orig_load = getattr(_t, '_orig_load', _t.load)
def _compat_load(*a, **k):
    k.setdefault('weights_only', False)
    return _t._orig_load(*a, **k)
_t.load = _compat_load
try:
    import matplotlib; matplotlib.rcParams['axes.unicode_minus'] = False
except Exception: pass
print('[colab-compat] torch.load weights_only=False 패치')


[colab-compat] torch.load weights_only=False 패치


In [3]:
# [로컬패치] aiffel env에 datasets/loralib/trl 이미 설치됨 → 재설치 생략
print('deps already installed in aiffel env')


deps already installed in aiffel env


In [4]:
# [로컬패치] KoChatGPT clone 및 chatgpt 패키지(colossalai 제거 패치본)는 로컬에 이미 준비됨
print('KoChatGPT & chatgpt ready at', r'/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build')


KoChatGPT & chatgpt ready at /home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build


In [6]:
import os

modifications = [
    {
        "file": "/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

⚠️ /home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/callbacks/save_checkpoint.py 파일의 3번째 줄이 예상과 다릅니다.
   예상: from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy
   실제: from chatgpt.trainer.strategies import Strategy
⚠️ /home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/callbacks/save_checkpoint.py 파일의 71번째 줄이 예상과 다릅니다.
   예상: only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)
   실제: only_rank0 = True  # NaiveStrategy 단일 GPU: 항상 rank0 저장
⚠️ /home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/callbacks/save_checkpoint.py 수정할 내용이 없습니다.
⚠️ /home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/strategies/__init__.py 파일의 1번째 줄이 예상과 다릅니다.
   예상: from .colossalai import ColossalAIStrategy
   실제: from .ddp import DDPStrategy
⚠️ /home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/chatgpt/trainer/strategies/__init__.py 수정할 내용이 없습니다.
⚠️ /home/gmw/Docu

In [7]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

Torch version:2.9.1+rocm7.2.1.gitff65f5bc
Cuda version: None
transformers version: 4.57.6
GPU 사용 가능여부: True


## 1. 베이스 모델 확인 — 학습 전에는 무엇을 하는 모델인가

RLHF 를 시작하기 전에, 아무것도 안 배운 상태의 모델이 무엇을 하는지 먼저 봐야 나중에 비교가 된다.

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "skt/ko-gpt-trinity-1.2B-v0.5"
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    'skt/ko-gpt-trinity-1.2B-v0.5',
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>')

# 가드: 클래스명이 아니라 '기능'으로 검증 (transformers 버전에 따라 클래스명이 흔들림)
_s = '불고기용 고기 한우에요?'
_rt = tokenizer.decode(tokenizer.encode(_s))
assert tokenizer.vocab_size == 51200, f"vocab={tokenizer.vocab_size}"
assert '�' not in _rt and '불고기' in _rt, repr(_rt)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


In [9]:
tokenizer.model_max_length

1000000000000000019884624838656

In [10]:
model.config.n_positions

1024

### SFT 전 생성 — 이 모델은 "이어쓰기 기계"다

시 한 구절을 넣어보면 이 모델의 정체가 드러난다.
**질문에 답하는 게 아니라 뒤에 올 법한 말을 잇는다.** 사전학습에서 배운 게 "다음 단어 맞히기" 뿐이라 그렇다.
그래서 여기에 질문을 던져봐야 답이 아니라 질문 뒤에 올 법한 문장이 나온다.

이 출력이 뒤에 나올 SFT 후 출력과의 비교 기준점이 된다.

In [11]:
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."

In [14]:
max_length=128
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
print(tokenizer.decode(output_greedy[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.


## 2. 데이터셋 3종 — 단계마다 사람 손이 들어가는 방식이 다르다

RLHF 3단계는 각각 다른 데이터를 쓴다. 그리고 **각 데이터에 사람이 개입한 방식이 다르다**는 게 핵심이다.

| 단계 | 데이터 | 사람이 한 일 |
|---|---|---|
| SFT | 질문-답변 쌍 12,000개 | **답을 직접 썼다** |
| RM | 한 질문에 답 3개 + ranking | **어느 게 나은지 골랐다** |
| PPO | 프롬프트만 | (모델이 스스로 답을 만든다) |

아래에서 셋을 차례로 열어본다.

In [18]:
import json
data_path_1_SFT = '/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

RM 데이터에는 **모범답안이 없다.** 한 질문에 답이 3개 있고 `ranking` 만 붙어 있다.
`ranking` 은 자리 순서대로 각 답의 등수이고 **숫자가 작을수록 좋은 답**이다.
(추측하지 않고 뒤에 나오는 쌍 생성 코드가 이 값을 어떻게 쓰는지 보고 확인했다.)

In [19]:
data_path_2_RM = '/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

PPO 데이터에는 **프롬프트만 있다.** 답이 아예 없다.
모델이 스스로 답을 만들고 RM 이 점수를 매기는 구조라 정답지가 필요 없기 때문이다.

In [20]:
data_path_3_PPO = '/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]

## 3. 1단계 SFT — 대답하는 "형식"을 가르친다

질문-답변 쌍을 보여주며 "이런 게 오면 이렇게 답해"를 시키는 단계다. 정답지를 쥐어주고 따라 하게 하므로
지도학습(Supervised)이다.

**여기서 가르치는 건 형식이지 지식이 아니다.** 데이터가 전부 반말이었다면 모델은 반말로 답했을 것이다.
그래서 SFT 를 마쳐도 사실오류는 그대로 남는다 — 뒤 생성 결과에서 실제로 확인된다.

`label[:source_len] = -100` 은 **질문 부분은 loss 계산에서 빼겠다**는 뜻이다(-100 은 파이토치
CrossEntropyLoss 의 ignore_index 기본값). 질문을 따라 쓰는 걸 배우면 안 되고 답변만 배워야 하니까.

In [21]:
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy

### 여기가 이 프로젝트의 개선 지점이다

베이스 모델을 `skt/kogpt2-base-v2`(117M) 에서 **`skt/ko-gpt-trinity-1.2B-v0.5`(1.2B)** 로 교체했다.
아래 3단계 전체가 이 교체된 모델 위에서 돌아간다.

같은 파이프라인을 kogpt2 로도 따로 돌려두었고(baseline), 두 결과를 6장에서 비교한다.

In [22]:
model = AutoModelForCausalLM.from_pretrained('skt/ko-gpt-trinity-1.2B-v0.5')
from transformers import PreTrainedTokenizerFast
# [수정] AutoTokenizer는 transformers 5.x에서 byte-level(slow) GPT2Tokenizer로 풀려
# 한글이 바이트 단위로 쪼개진 ID로 학습됨 → 생성 결과가 깨지는 원인.
# fast 토크나이저로 고정해 학습~생성 전 구간의 ID 체계를 일치시킨다.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    'skt/ko-gpt-trinity-1.2B-v0.5', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,
)

print(tokenizer)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


PreTrainedTokenizerFast(name_or_path='skt/ko-gpt-trinity-1.2B-v0.5', vocab_size=51200, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '</s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<usr>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<sys>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<mask>", 

In [23]:

# ── SFT 학습용 데이터셋 클래스 ──
# 하는 일: (질문, 답변) 쌍이 담긴 jsonl 을 읽어서, 모델이 먹을 수 있는 토큰 id 묶음으로 바꾼다.
# 핵심은 맨 아래 label[:source_len] = -100 이다. "질문 부분은 채점하지 마라"는 뜻.
class SFT_dataset(Dataset):

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        # 1) 원본 데이터 로드 : [{"prompt": 질문, "completion": 답변}, ...] 형태
        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        # 2) 질문을 항상 같은 틀에 끼워 넣는다.
        #    모델은 이 고정된 틀을 보고 "### Response(응답): 다음에는 답이 온다"를 배우게 된다.
        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        # 3) sources = 틀에 끼운 질문까지만 (답 없음)
        sources = []
        for example in list_data_dict:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        # 4) targets = 답변 + eos(문장 끝 표시). eos 가 있어야 모델이 "여기서 그만"을 배운다.
        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        # 5) examples = 질문 + 답변 을 한 줄로 이어붙인 것. 실제로 모델에 들어가는 문장이다.
        examples = [s + t for s, t in zip(sources, targets)]

        # 6) 두 벌을 각각 토크나이즈한다.
        #    sources 쪽을 따로 재는 이유는 "질문이 몇 토큰까지인지" 길이를 알아야 하기 때문이다.
        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        # 7) 입력은 질문+답변 전체
        input_ids = examples_tokenized["input_ids"]
        # 8) 정답표(label)는 입력을 그대로 복사한 뒤, 앞쪽 질문 구간만 -100 으로 덮는다.
        #    -100 은 파이토치 CrossEntropyLoss 의 ignore_index 기본값이라 loss 계산에서 통째로 빠진다.
        #    ★즉 "질문을 따라 쓰는 것"은 안 배우고 "답변을 쓰는 것"만 배운다. 이게 SFT 의 요령이다.
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        # 문장 리스트를 하나씩 토크나이즈한다. 너무 길면 자른다(truncation).
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        # 실제 길이 = 패딩(PAD)이 아닌 토큰의 개수. 위 8)에서 "질문이 어디까지인지" 자를 때 쓴다.
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        # 데이터 개수 (Dataset 규약)
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        # i번째 샘플 하나를 꺼낸다 (Dataset 규약)
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])


In [24]:
# ── 배치를 만드는 collator ──
# 위 Dataset 은 문장을 하나씩 내주는데, 학습은 여러 개를 묶어(batch) 돌린다.
# 문장 길이가 제각각이라 직사각형으로 못 만드니, 짧은 걸 채워서(padding) 길이를 맞추는 게 이 클래스의 일이다.
@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        # 배치 안의 샘플들에서 input_ids 묶음, labels 묶음을 각각 뽑는다.
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        # 입력은 PAD 토큰으로 채운다. batch_first=True -> (배치, 길이) 모양.
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        # ★정답표는 PAD 가 아니라 -100 으로 채운다. 채운 칸까지 채점하면 안 되니까.
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            # attention_mask = PAD 가 아닌 자리만 True. 모델이 채운 칸을 안 쳐다보게 만든다.
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )


In [25]:
# ── 실제로 데이터셋과 collator 인스턴스를 만든다 ──
train_dataset = SFT_dataset(data_path_1_SFT='/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl', tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

# 첫 샘플을 찍어 확인한다.
# input 은 질문+답변 전체 토큰이고, output(=label) 은 앞쪽이 전부 -100 으로 덮여 있어야 정상이다.
# -100 이 쭉 이어지다가 숫자가 나오는 지점부터가 "채점 대상 = 답변"이다.
print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])


input : tensor([30132, 42872, 33313, 30679, 40479, 39911,   384, 22509, 21921, 25372,
          385, 31245, 23280, 34957, 25617, 36539, 29991, 25624, 25400, 31167,
          376, 42872,   379, 46803,   456, 30303, 35353,   384, 25785, 20573,
        37780,   383, 46900, 43226,   565, 27071, 23151, 31555, 41690, 35071,
        25400, 31269, 32677, 30765, 31810, 36229, 30326, 33889, 30093, 34957,
        25617, 30021, 30434, 29991, 39687, 34036, 19016, 31997, 49906, 19352,
        30011, 30904, 36731, 43502, 30228, 31214, 30326, 29991, 31621, 33314,
        34347, 30843, 50342, 33512, 31370, 34243, 29991, 35144, 32586, 32622,
        44680, 30110, 21844, 39826, 34803, 31356, 39075, 30242, 36966, 29985,
        34179, 36513, 30718, 35557, 32361, 31018, 29404, 35942, 19352, 41049,
            1])
output: tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -1

In [27]:
# ── 학습 설정과 Trainer 조립 ──
# 허깅페이스 Trainer 는 "설정(TrainingArguments) + 모델 + 데이터 + collator" 를 받아 학습 루프를 대신 돌려준다.
training_args = transformers.TrainingArguments(
    output_dir="/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out_trinity/test",
    num_train_epochs=1,               # 1에폭만. 노드 기본값이고, 형식을 익히는 데는 이 정도로도 된다
    per_device_train_batch_size=8,    # 한 번에 8문장씩
    per_device_eval_batch_size=8,
    warmup_steps=5,                   # 처음 5스텝은 학습률을 서서히 올린다(초반 흔들림 방지)
    prediction_loss_only=True,        # 평가 때 loss 만 본다(생성까지 안 함 -> 빠르다)
    fp16 = True                       # 16비트 혼합정밀. 메모리 절약 + 속도
    )
trainer = transformers.Trainer(
    model=model,                      # 위에서 교체한 trinity 모델
    args=training_args,
    data_collator=data_collator,      # 배치 만들 때 패딩 담당
    train_dataset=train_dataset
)


2026-07-21 16:20:39.188383: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 16:20:39.189273: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-21 16:20:39.193443: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-21 16:20:39.203909: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784618439.217871  693428 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784618439.22

SFT 학습을 돌린다. loss 가 내려가는지 본다.

In [28]:
trainer.train()
model.save_pretrained('/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out_trinity/output_1_SFT')

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.559000
1000,2.406900
1500,2.309500


### SFT 후 생성 — 형식이 생겼다

이제 `### Instruction ... ### Response` 형식을 따라 답하는 게 눈에 보인다.
1장에서 시를 이어 쓰던 모델과 같은 모델이다. **대답하는 자세를 배운 것이다.**

다만 내용은 여전히 틀릴 수 있다. 형식과 사실은 다른 문제다.

In [29]:
generator = transformers.pipeline('text-generation', model='/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out_trinity/output_1_SFT', tokenizer=tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=375, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

Device set to use cuda:0



### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 AI 어시스턴트이기 때문에 불고기용 고기의 종류나 원산지를 알 수 없습니다. 하지만 일반적으로 불고기용은 한우, 돼지고기, 소고기 등 다양한 종류의 고기를 사용하며, 한우는 대표적인 고급 육류 중 하나입니다. \n\n하지만 정확한 정보를 얻으시려면 해당 식당의

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'1967년입니다. 이 당시 부통령은 리처드 닉슨(Richard Nixon)이었습니다. \n 따라서, 리처드 닉슨은 43대 부통령을 역임한 적이 없습니다.  하지만 리처드 닉슨은 1952년부터 1954년까지

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어?

### Response(응답):'저는 인공지능 어시스턴트이므로, 시카고 오헤어 공항이 어디에 위치해 있는지 알 수 없습니다. 추가적인 정보를 제공해주시면 더 정확한 답변을 드릴 수 있을 것 같습니다. 감사합니다.  \n\n감사합니다! 
 }

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'미세먼지 농도는 어제와 비교해서 개선되었지만 아직도 나쁜 수준입니다. 마스크 착용과 실외 활동 자제를 권장합니다. 또한, 실내 공기 질 개선을 위해 환기 및 청소를 자주 하시는 것이 좋습니다.  \n\n마스크를 착용하고 실외 활동을 자제하는


In [30]:
torch.cuda.empty_cache()

## 4. 2단계 RM — 어느 답이 더 나은지 점수를 매기는 심판

SFT 로는 못 가르치는 게 있다. **좋은 답은 정답이 하나가 아니다.**
같은 질문에 형식이 맞는 답이 여러 개일 때 "이 중 뭐가 낫냐"는 정답지에 순위가 없어서 가르칠 수 없다.

그래서 **선호(preference)를 배운 심판**을 따로 기른다. 그게 Reward Model 이다.

In [31]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

import random

RM 은 언어모델 뒤에 **스칼라 하나를 뱉는 선형층**을 붙인 형태다.
`nn.Linear(model.config.n_embd, 1)` 의 출력 1이 곧 "이 문장 몇 점"이다.

In [32]:
# ── 보상모델(RM) 클래스 ──
# 구조를 한 줄로 말하면 "언어모델 몸통 + 점수 하나를 뱉는 머리" 다.
# 언어모델은 원래 다음 단어의 확률을 뱉는데, RM 은 그 대신 "이 문장 몇 점"이라는 숫자 하나를 뱉어야 한다.
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        # 1) 몸통 준비 : 저장된 모델이 있으면 불러오고, 없으면 설정만으로 새로 만든다.
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            # 토크나이저에 특수토큰을 추가했다면 임베딩 표의 행 수도 같이 늘려줘야 한다.
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            # 메모리를 아끼는 대신 계산을 다시 하는 옵션 (여기선 안 씀)
            model.gradient_checkpointing_enable()

        # 2) ★머리 : 은닉벡터(n_embd 차원)를 받아 숫자 1개로 줄이는 선형층.
        #    출력이 1이라는 게 곧 "점수 하나"라는 뜻이다. 이게 RM 을 언어모델과 가르는 지점이다.
        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        # 저장할 땐 몸통만 저장한다 (PPO 단계에서 critic 이 이걸 불러 쓴다)
        if self.pretrained is not None:
            self.model.save_pretrained(dir)


In [33]:
model = AutoModelForCausalLM.from_pretrained('skt/ko-gpt-trinity-1.2B-v0.5')
from transformers import PreTrainedTokenizerFast
# [수정] AutoTokenizer는 transformers 5.x에서 byte-level(slow) GPT2Tokenizer로 풀려
# 한글이 바이트 단위로 쪼개진 ID로 학습됨 → 생성 결과가 깨지는 원인.
# fast 토크나이저로 고정해 학습~생성 전 구간의 ID 체계를 일치시킨다.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    'skt/ko-gpt-trinity-1.2B-v0.5', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,
)

with NaiveStrategy().model_init_context():
        model = GPTRM_custom(pretrained='skt/ko-gpt-trinity-1.2B-v0.5', lora_rank=0, tokenizer=tokenizer).cuda()

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


### 비교를 쌍으로 바꾼다

답 3개로 가능한 짝 3개를 만들고, 각 짝에서 등수가 낮은(=더 좋은) 쪽을 `chosen`,
높은 쪽을 `rejected` 로 붙인다. `ranking[0] < ranking[1]` 로 대소를 비교하는 이 코드가
곧 "ranking 숫자가 작을수록 좋은 답"이라는 근거다.

In [34]:
# ── RM 데이터를 "이긴 답 vs 진 답" 쌍으로 바꾼다 ──
# 원본 한 줄은 [질문 1개 + 답 3개 + ranking] 이다. ranking 은 자리 순서대로 각 답의 등수이고 숫자가 작을수록 좋다.
# 그런데 RM 의 손실 함수는 한 번에 두 개만 비교한다(이긴 쪽 점수 - 진 쪽 점수).
# 그래서 답 3개로 만들 수 있는 짝 3개 (0vs1, 0vs2, 1vs2) 로 펼쳐준다. -> 데이터가 3배가 된다.
with open('/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    # 짝 (1) completion_0 vs completion_1 : 등수가 작은(=좋은) 쪽이 chosen
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    # 짝 (2) completion_0 vs completion_2
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    # 짝 (3) completion_1 vs completion_2
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)



    total_data_ranking2chosen.extend(one_data_ranking2chosen)

# 개수가 정확히 3배가 되는지 확인한다 (질문 1개 -> 쌍 3개)
print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
# 실제로 어떻게 생겼는지 하나 찍어본다. prompt / chosen / rejected 세 칸이면 정상.
print('data example: \n%s'%total_data_ranking2chosen[45])


before data num: 10220
after  data num: 30660
data example: 
{'prompt': '애플은 리사를 어떻게 처리했어', 'chosen': '애플이 누구인지 명확히 알 수 없어서, 리사가 누구인지와 어떤 상황에서 처리되었는지에 대한 추가적인 정보가 필요합니다. 따라서, 보다 정확한 답변을 제공할 수 없습니다.', 'rejected': '애플은 리사를 위해 고객 서비스 부서에서 고객 다양한 컴퓨터 관련 문제에 대해 응답하는 데 필요한 모든 지원을 제공했습니다. 사용자가 하드웨어 문제를 경험할 때, 전문가들은 필요한 수리(수리, 추가 부품 제공, 소프트웨어 업그레이드 등)을 제공해 드릴 수 있습니다. 또한, 사용자가 사용 방법 문제나 기타 문제를 경험할 때, 대화 상대로 사용자를 지원할 수 있는 전문 고객 서비스 직원들이 사용자에게 상담하고 도움을 주는 데 도움이 될 수 있는 정보를 제공합니다. 또한, 인터넷에서 제공되는 정보를 통해 문제를 해결하거나 고객 서비스 웹 사이트를 통해 자신의 문제를 진단할 수 있도록 하는 등 다양한 방법으로 리사를 처리해 왔습니다.'}


In [35]:
import random
random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[45])

{'prompt': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은?', 'chosen': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은 류승완의 사무실입니다.', 'rejected': '대구 영화사옥'}


In [36]:
# ── 학습셋/검증셋으로 나눈다 ──
# 위에서 만든 쌍 데이터 중 앞 1000개만 학습에 쓴다. (노드 기본값. 전부 쓰면 시간이 너무 걸린다)
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]     # 그 다음 200개는 검증용

print(len(train_data))
print(len(eval_data))

# RewardDataset 이 (prompt+chosen), (prompt+rejected) 두 벌을 각각 토크나이즈해 들고 있게 된다.
# 512 는 최대 토큰 길이.
train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)


1000
200


> 학습 진행바의 최종 상태 (아래 "읽기 전에" 참고).

```
100% 1000/1000 [00:00<00:00, 1337.59it/s]
100% 200/200 [00:00<00:00, 1280.03it/s]
```


In [37]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
흑고래의 무게는 어느 정도야
######################################################################
## chosen ##
흑고래의 평균 몸무게는 약 25~40톤 정도이지만, 최대 몸무게는 50톤 이상에 이를 수 있습니다.
######################################################################
## rejected ##
흑고래의 무게는 매우 다양하게 달라집니다. 약 200kg에서 10톤까지 달라질 수 있습니다.


### RM 의 손실 함수

```python
probs = torch.sigmoid(chosen_reward - reject_reward)
loss  = -torch.log(probs).mean()
```

**"이긴 답 점수에서 진 답 점수를 뺀 값이 커지게 하라"** 이다.
절대 점수가 몇 점이어야 하는지는 아무도 안 알려준다. **순서만 맞으면 된다.**

읽는 눈금이 하나 있다. 두 점수가 완전히 같으면 `sigmoid(0)=0.5`, `-log(0.5)=0.693` 이다.
즉 **loss 0.693 이 "좋은 답과 나쁜 답을 전혀 구별 못 하는" 동전던지기 지점**이다.

In [38]:
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=1)

In [39]:
trainer.fit(use_lora=0)

model.save_pretrained('/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out_trinity/output_2_RM')

> 학습 진행바의 최종 상태 (아래 "읽기 전에" 참고).

```
Train epoch: 100% 1/1 [3:17:26<00:00, 7944.62s/it]
Train step of epoch 0: 100% 250/250 [2:12:24<00:00, 27.74s/it, loss=0.624, dist_mean=0.254]
```

**읽어보면**: RM 학습은 250스텝을 2시간 12분에 걸쳐 돌았고 **최종 loss 0.624** 로 끝났다.

이 숫자를 읽는 눈금이 있다. 두 점수가 완전히 같으면 `sigmoid(0)=0.5`, `-log(0.5)=0.693` 이므로
**0.693 이 "좋은 답과 나쁜 답을 전혀 구별 못 하는" 동전던지기 지점**이다.
0.624 는 거기서 **조금 내려온 정도**다. 심판이 눈을 뜨긴 했지만 많이 뜬 건 아니라는 뜻이고,
이게 뒤에서 RM 점수를 찍어봤을 때 나오는 결과와 이어진다.


### RM 이 실제로 잘 배웠는지 점수를 찍어본다

문장 네 개를 넣어 점수를 본다. **이 결과가 뒤(7장 자체평가)에서 중요한 근거가 된다.**

In [40]:
# ── 학습된 RM 에게 실제로 점수를 물어보는 함수 ──
def inference_RM(input_text):
    # 문장을 토큰 id 로 바꿔 GPU 로 올린다
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    # RM 에 통과시키면 (언어모델처럼 단어 확률이 아니라) 점수 하나가 나온다
    output = model(input_ids)
    # 텐서를 파이썬 숫자로 꺼낸다. detach() 는 "이건 학습에 안 쓴다"는 표시.
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

# ★일부러 나쁜 문장을 먼저 넣어본다. 심판이 제대로 배웠다면 낮은 점수가 나와야 한다.
#   (아래 셀들에서 좋은 문장도 넣어보고, 점수가 갈리는지 비교한다)
input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)


input: 인공지능은 똥멍청이 입니다
reward score: 0.5


In [41]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: 0.5


In [42]:
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다.
reward score: 0.5


In [43]:
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: 0.7


In [44]:
torch.cuda.empty_cache()

## 5. 3단계 PPO — 점수를 쫓되, 예전의 자신에게서 멀어지지 않게

이제 정답지를 안 쓴다. 모델이 답을 만들고 RM 이 점수를 매기고, 점수가 높아지는 쪽으로 민다.
그래서 이 단계가 지도학습이 아니라 강화학습이다.

In [45]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

### 모델이 네 개 등장한다

- `actor` — 학습시킬 본체. **1단계 SFT 모델**에서 출발한다.
- `critic` — 기대 점수(baseline)를 예측하는 보조. **2단계 RM** 에서 출발한다.
- `reward_model` — 심판. 얼려둔다.
- `initial_model` — **PPO 직전의 actor 를 통째로 복사해 얼려둔 것.**

마지막 것이 KL penalty 의 정체다. 점수만 쫓게 두면 모델은 "좋은 답"이 아니라
**"심판이 높은 점수를 주는 것"** 을 찾아간다(보상 해킹). 그래서 매 스텝
"얼려둔 예전 너라면 이런 말을 했겠나"를 묻고 멀어진 만큼 벌점을 뺀다.

PPO 의 P 가 Proximal(가까운) 인 것도 같은 뜻이다. 한 걸음에 멀리 가지 말라는 것.

In [46]:
# ── PPO 에 등장하는 모델 네 개를 만든다 ──
# 여기가 PPO 코드에서 제일 헷갈리는 곳이라 하나씩 적어둔다.
with NaiveStrategy().model_init_context():
    # (1) actor  = 학습시킬 본체. 1단계 SFT 결과에서 출발한다.
    actor = GPTActor(pretrained='/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out_trinity/output_1_SFT', lora_rank=0).to(torch.cuda.current_device())
    # (2) critic = 기대 점수(baseline)를 예측하는 보조. 2단계 RM 결과에서 출발한다.
    #     점수 r 만으로는 밀 방향을 못 정하고, "이 정도 상황이면 평균 몇 점"을 알아야 신호가 되기 때문이다.
    critic = GPTCritic(pretrained='/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out_trinity/output_2_RM', lora_rank=0).to(torch.cuda.current_device())

    # [수정] fast 토크나이저로 고정 (학습~생성 ID 체계 일치)
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        'skt/ko-gpt-trinity-1.2B-v0.5', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
        padding_side="right",
        model_max_length=512
    )

    # (3) initial_model = ★지금 actor 를 통째로 복사해 얼려둔 것 = "PPO 직전의 나".
    #     학습 내내 옆에 두고 "예전 너라면 이런 말을 했겠나"를 물어, 멀어진 만큼 벌점을 뺀다(KL penalty).
    #     이게 없으면 모델이 "좋은 답"이 아니라 "심판이 점수 주는 것"을 찾아가 망가진다(보상 해킹).
    initial_model = deepcopy(actor)
    # (4) reward_model = 심판. critic 의 몸통과 점수 머리를 복사해 만든다. 학습하지 않고 채점만 한다.
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


In [47]:
# ── 옵티마이저 : 학습되는 두 모델(actor, critic)에만 붙인다 ──
# reward_model 과 initial_model 은 얼려둔 것이라 옵티마이저가 없다. 여기서 그게 드러난다.
# 학습률 5e-6 은 아주 작은 값이다. PPO 는 한 걸음에 멀리 가면 무너져서 조심스럽게 민다.
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)


In [48]:
# ── 전략(strategy)에 모델과 옵티마이저를 등록한다 ──
# 원래는 여러 GPU 에 나눠 얹는 단계인데, 여기서는 NaiveStrategy(단일 GPU)라 사실상 그대로 돌려받는다.
# (원본은 colossalai 를 쓰지만 로컬에서 안 돌아 그 의존을 걷어내고 NaiveStrategy 만 남겼다)
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)


In [49]:
# ── PPO 학습에 쓸 데이터 : 질문(프롬프트)만 있다 ──
# ★SFT 데이터와 결정적으로 다른 점이다. 여기엔 정답이 없다.
#   모델이 스스로 답을 만들고 심판이 점수를 매기는 구조라 정답지가 필요 없기 때문이다.
with open('/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

# 학습 루프가 매번 호출할 토크나이즈 함수. 길이 96 으로 맞추고 GPU 로 올려서 돌려준다.
def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}


In [50]:
print(tokenize_fn('It takes something more than intelligence to act intelligently.'))

{'input_ids': tensor([[46390, 31369, 33712, 30541, 31338, 41607, 30586, 31024, 31482, 37404,
         31035, 30316, 32131,   460, 34763, 32017, 37762, 33441,   565, 37205,
         32131,   460, 34763, 32017, 31561, 36271,   390]], device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0')}


In [51]:
len(list_prompt)

12000

In [52]:
# ── PPO 학습기 조립 : 위에서 만든 네 모델을 한 자리에 모은다 ──
trainer = PPOTrainer(NaiveStrategy(),
                     actor,             # 학습되는 본체
                     critic,            # 학습되는 baseline 예측기
                     reward_model,      # 얼림 - 채점
                     initial_model,     # 얼림 - KL 기준점("예전의 나")
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,    # 생성 최대 길이
                     # 아래 셋은 답을 '만들 때'(rollout) 쓰는 옵션이다.
                     do_sample=True,    # 매번 같은 답이 아니라 확률적으로 뽑는다
                     temperature=1.0,   # 1.0 = 원래 확률 그대로
                     top_k=50,          # 후보를 상위 50개로 좁힌다
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)


PPO 학습을 돌린다. 노드 기본값대로 `num_episodes=10` 이다.
**이 값이 작다는 게 뒤에서 PPO 결과를 해석할 때 중요해진다.**

In [53]:
# ── PPO 학습 실행 ──
# num_episodes=10   : 에피소드 10회. ★이 값이 작다는 게 뒤에서 결과를 해석할 때 중요해진다.
#                     (실제 서비스급은 이걸 수만 번 돌린다)
# max_timesteps=3   : 에피소드 하나에서 답을 3번 만든다(경험 쌓기)
# update_timesteps=3: 경험 3개가 모이면 모델을 한 번 업데이트한다
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

# ★actor 가 아니라 actor.model 을 저장한다.
#   actor 는 강화학습용 래퍼라 그대로 저장하면 나중에 불러 쓸 때 형식이 안 맞는다.
#   안쪽의 실제 언어모델(.model)을 꺼내 저장해야 한다.
actor.model.save_pretrained('/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build/nb_out_trinity/output_3_PPO')


> 학습 진행바의 최종 상태 (아래 "읽기 전에" 참고).

```
Episode [1/10]: 100% 3/3 [15:41<00:00, 319.28s/it]
Train epoch [1/1]: 100% 3/3 [00:42<00:00, 14.01s/it, actor_loss=0, critic_loss=0.0152]
Episode [2/10]: 100% 3/3 [15:38<00:00, 318.00s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.85s/it, actor_loss=0.192, critic_loss=0.0881]
Episode [3/10]: 100% 3/3 [15:43<00:00, 319.43s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.86s/it, actor_loss=0.255, critic_loss=0.0163]
Episode [4/10]: 100% 3/3 [15:41<00:00, 318.78s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.84s/it, actor_loss=-0.195, critic_loss=0.0402]
Episode [5/10]: 100% 3/3 [15:40<00:00, 318.96s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.85s/it, actor_loss=-0.127, critic_loss=0.00335]
Episode [6/10]: 100% 3/3 [15:41<00:00, 319.27s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.85s/it, actor_loss=0.154, critic_loss=0.0239]
Episode [7/10]: 100% 3/3 [15:41<00:00, 318.86s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.80s/it, actor_loss=0.147, critic_loss=0.000951]
Episode [8/10]: 100% 3/3 [15:38<00:00, 317.94s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.81s/it, actor_loss=-0.0929, critic_loss=0.0138]
Episode [9/10]: 100% 3/3 [14:57<00:00, 304.05s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.82s/it, actor_loss=-0.0424, critic_loss=0.017]
Episode [10/10]: 100% 3/3 [14:48<00:00, 295.13s/it]
Train epoch [1/1]: 100% 3/3 [00:41<00:00, 13.84s/it, actor_loss=0.145, critic_loss=0.00456]
```

**읽어보면 — 이게 이 프로젝트에서 제일 정직하게 남겨야 할 기록이다.**

에피소드 10회 동안 `actor_loss` 가 이렇게 움직였다.

```
0  ->  0.192  ->  0.255  ->  -0.195  ->  -0.127  ->  0.154  ->  0.147  ->  -0.0929  ->  -0.0424  ->  0.145
```

**방향이 없다.** 줄어들지도 늘어나지도 않고 -0.195 에서 0.255 사이를 그냥 오간다.
(PPO 의 actor_loss 는 부호 자체가 업데이트 방향이라 0으로 수렴해야 하는 값은 아니지만,
**10회 동안 아무 추세도 안 보인다는 건 학습이 진행됐다고 말할 근거가 없다는 뜻**이다.)

그리고 시간을 보면 에피소드 하나에 약 15분 40초씩, 총 2시간 40분 가까이 걸렸다.
**그 시간의 대부분이 학습이 아니라 생성(rollout)이다** — 학습(`Train epoch`)은 매번 41초뿐이고
나머지가 답을 만드는 시간이다.

그러니까 이 단계는 **"오래 돌렸는데 조금 배웠다"가 아니라 "거의 안 배웠다"** 에 가깝다.
뒤 생성 결과에서 PPO 쪽이 SFT 보다 나아지지 않는 것도 이것과 앞뒤가 맞는다.


### 3단계를 다 거친 모델의 생성

생성 직전에 토크나이저를 다시 고정한다. 앞 단계에서 `AutoTokenizer` 로 여러 번 재할당되는데,
그대로 두면 다른 토크나이저가 잡혀 한글이 깨질 수 있어서다. (실제로 겪고 나서 넣은 가드다.)

여기까지가 학습이고, 다음 장부터가 **개선 비교분석**이다.

In [54]:
# ── 토크나이저: 생성 직전 재로드 + 기능 가드 (한글 깨짐 방지) ──
# 앞선 SFT/RM/PPO 셀에서 tokenizer가 AutoTokenizer로 여러 번 재할당되므로,
# 생성 직전에 KoGPT2 fast 토크나이저로 다시 고정한다.
from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    'skt/ko-gpt-trinity-1.2B-v0.5',
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>')

# 위 로드가 여전히 GPT-2(vocab 50257)로 가로채이면 아래 주입 방식으로 교체:
# from huggingface_hub import hf_hub_download
# from tokenizers import Tokenizer
# core = Tokenizer.from_file(hf_hub_download('skt/ko-gpt-trinity-1.2B-v0.5', 'tokenizer.json'))
# tokenizer = PreTrainedTokenizerFast(tokenizer_object=core,
#     bos_token='</s>', eos_token='</s>', unk_token='<unk>',
#     pad_token='<pad>', mask_token='<mask>')

_s = '불고기용 고기 한우에요?'
_rt = tokenizer.decode(tokenizer.encode(_s))
assert tokenizer.vocab_size == 51200, f"vocab={tokenizer.vocab_size}"
assert '�' not in _rt and '불고기' in _rt, repr(_rt)
print('[tokenizer-guard] KoGPT2 OK - vocab', tokenizer.vocab_size)


def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1,
                             bad_words_ids=None,
                             pad_token_id=tokenizer.pad_token_id,
                             eos_token_id=tokenizer.eos_token_id)

    # BPE 계열은 clean_up_tokenization_spaces=False 가 안전 (True는 무시/파괴적)
    output = tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)

    print(f"\n결과: {output}")
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor.model)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


[tokenizer-guard] KoGPT2 OK - vocab 51200



결과: ### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'네, 불고기용 고기는 한우의 갈비 부위로 만든 양념으로 만든 한우를 의미합니다.



결과: ### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨이 43대 부통령직을 수행한 년도는 1997년이다.



결과: ### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어

### Response(응답):'시카고 오헤어 국제공항은 미국 시카고 주 오헤어 주 포투갈의 도시 포투갈의 시카고 시로고스 공항( City of George County Portugal) 일대에 위치해 있습니다. 공항에서 약 25km 떨어진 포투갈의 중심지에 위치하고 있습니다. 공항의 수용 능력은 약 5만 명입니다.



결과: ### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'오늘 미세먼지 농도가 높아졌습니다. 미세먼지 농도가 높아지면 건강에 좋지 않을 수 있으며, 실외 활동과 조업 등의 환경적 요인에 따라 달라질 수 있습니다. 또한, 미세먼지는 우리 생활 속에서의 대기 오염, 미세먼지 발생 원인 등에 따라 다양한 요인에 의해 영향을 받을 수 있습니다. \n\n먼저, 실내 공기질은 오염도에 따라 다릅니다. 실내 공기질은 알레르기, 천식, 미세먼지 등 다양한 요인에 따라 영향을 받습니다. 미세먼지가 발생하기 쉬운 환경에서는 공기정화기, 공기청정기 등의 자연 필터나 환기 제품을 사용하는 것이 좋습니다. 또한, 미세먼지 농도가 높을 때는 실외 활동이나 조업 시 공기의 질이 낮을 수 있으므로 주의가 필요합니다. \n\n조업환경에서도 미세먼지 농도는 매우 높았습니다. 공장의 화학물질, 오염된 물질 등에 의한 공기 오염도 그 이유 중 하나입니다.\n\n환기와 세탁 등의 환경적 요인에 따라 공기 질을 개선하는 데에도 영향을 줄 수 있습니다. 또한, 실내 공기 질은 주변 환경, 생활습관 등의 다양한 요인에 의해 영향을 받기 때문에, 실내 공기 질 개선을 위해서는 환경과 생활습관 개선


## 6. 개선 비교분석 — baseline KoGPT2 vs 모델 교체(ko-gpt-trinity-1.2B)

여기부터는 내가 이 노드에서 잡은 **개선 축**을 확인하는 부분이다.
위(앞 셀들)에서는 `skt/ko-gpt-trinity-1.2B-v0.5` 로 SFT -> RM -> PPO 3단계를 다 돌렸다.
그럼 "**그냥 KoGPT2(baseline)로 했을 때보다 정말 나아졌나?**" 를 눈으로도, 숫자로도 봐야 한다.

그래서 아래에서:
1. **정성 비교** — 같은 질문을 두 모델에 똑같이 던져 답을 나란히 본다.
2. **정량 비교** — SFT 데이터의 정답(completion)을 참조로 **BLEU / ROUGE** 를 잰다.

### ★ 정직하게 짚는 한계 (중요)
- 이 노드 원본은 SFT 데이터 12,000개를 **전부 학습에 썼다.** 그래서 완전한 held-out(학습에 안 쓴 평가 전용) 셋이 없다.
- 따라서 아래 BLEU/ROUGE 는 **학습에 포함된 데이터로 잰 값(train 참고치)** 이라, "이 모델의 절대 성능"이 아니다.
- 다만 **두 모델(kogpt2 vs trinity)이 똑같은 데이터·똑같은 조건으로 평가**되므로, **둘을 견주는 상대 비교로는 공정**하다. 그 목적으로만 읽는다.


In [1]:
# ── 준비: import + 경로/설정 ──
# 이 셀부터는 앞 학습 셀과 독립이다. (커널을 새로 봐도 되게 자기완결로 다시 import 한다.)
import json, torch, transformers
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
from konlpy.tag import Okt                       # 한국어 형태소 분석기 (BLEU/ROUGE 토큰 단위)
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

torch.manual_seed(42)                            # 생성에 do_sample 이 있어 흔들리므로 seed 고정(재현성)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV, '| torch', torch.__version__)

BASE = '/home/gmw/Documents/AIFFEL_Work/_scratch/05_LLM/LLM_RLHF/build'

# 네 모델의 저장 경로 (앞 셀들이 학습해 저장해 둔 것)
PATHS = {
    'kogpt2_SFT':  BASE + '/nb_out/output_1_SFT',
    'kogpt2_PPO':  BASE + '/nb_out/output_3_PPO',
    'trinity_SFT': BASE + '/nb_out_trinity/output_1_SFT',
    'trinity_PPO': BASE + '/nb_out_trinity/output_3_PPO',
}
# 각 모델이 쓰는 토크나이저 (vocab 은 둘 다 51200 으로 같지만, 이름은 모델별로 맞춘다)
TOK_NAME = {'kogpt2': 'skt/kogpt2-base-v2', 'trinity': 'skt/ko-gpt-trinity-1.2B-v0.5'}

N_EVAL = 80          # 정량 평가에 쓸 프롬프트 수 (SFT 데이터 뒤쪽에서 고정으로 뗀다)


device: cuda | torch 2.9.1+rocm7.2.1.gitff65f5bc


In [2]:
# ── 공통 유틸: 토크나이저 로드 / 생성 / 지표 함수 ──
def load_tokenizer(kind):
    # kind = 'kogpt2' or 'trinity'. 앞 셀들과 똑같은 인자로 로드한다.
    return PreTrainedTokenizerFast.from_pretrained(
        TOK_NAME[kind], bos_token='</s>', eos_token='</s>',
        unk_token='<unk>', pad_token='<pad>', mask_token='<mask>')

PROMPT_TMPL = "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"

def make_prompt(q):
    return PROMPT_TMPL.format(prompt=q)

def strip_response(generated_text):
    # 생성 결과에서 "### Response(응답):" 뒤(=실제 답변)만 잘라낸다.
    key = "### Response(응답):"
    if key in generated_text:
        ans = generated_text.split(key, 1)[1]
    else:
        ans = generated_text
    # 혹시 다음 Instruction 이 이어지면 거기서 끊는다.
    ans = ans.split("### Instruction", 1)[0]
    return ans.strip().strip("'\"").strip()

def clean_ref(completion):
    # 참조 정답(completion)도 양끝 따옴표/공백을 정리해 비교 공정하게.
    return completion.strip().strip("'\"").strip()

# SFT 단계 생성 (앞 셀 29 관례 그대로: beam4 + repetition penalty)
SFT_ARGS = dict(num_beams=4, repetition_penalty=2.0, no_repeat_ngram_size=4,
                eos_token_id=375, max_new_tokens=64, do_sample=True, top_k=50,
                early_stopping=True)

def gen_sft(model_path, tokenizer, questions):
    gen = transformers.pipeline('text-generation', model=model_path,
                                tokenizer=tokenizer, device=0 if DEV=='cuda' else -1)
    prompts = [make_prompt(q) for q in questions]
    outs = gen(prompts, **SFT_ARGS)
    return [strip_response(o[0]['generated_text']) for o in outs]

# PPO 단계 생성 (앞 셀 54 관례 그대로: sampling)
def gen_ppo(model, tokenizer, questions):
    res = []
    for q in questions:
        ids = tokenizer.encode(make_prompt(q), return_tensors='pt').to(DEV)
        out = model.generate(ids, max_length=250, do_sample=True, top_k=50, top_p=0.95,
                             num_return_sequences=1, pad_token_id=tokenizer.pad_token_id,
                             eos_token_id=tokenizer.eos_token_id)
        txt = tokenizer.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
        res.append(strip_response(txt))
    return res

# ── 지표: 한국어라 형태소 단위로 BLEU/ROUGE ──
_okt = Okt()
def morphs(text):
    return _okt.morphs(text)

class OktTokenizer:                       # rouge_score 가 요구하는 .tokenize(text) 인터페이스
    def tokenize(self, text):
        return _okt.morphs(text)

_smooth = SmoothingFunction().method1
def bleu(ref, hyp):
    r, h = morphs(ref), morphs(hyp)
    if len(h) == 0:
        return 0.0
    return sentence_bleu([r], h, smoothing_function=_smooth)

_rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False, tokenizer=OktTokenizer())
def rouge_lf(ref, hyp):
    s = _rouge.score(ref, hyp)
    return s['rouge1'].fmeasure, s['rougeL'].fmeasure

print('유틸 준비 완료')


유틸 준비 완료


In [3]:
# ── 평가셋 로드 ──
# SFT 데이터(prompt, completion 쌍)에서 뒤쪽 N_EVAL 개를 고정으로 뗀다.
# (무작위 아님 = 재현 가능. 앞에서 학습에 이미 썼으니 held-out 아니라 train 참고치임을 기억.)
sft_path = BASE + '/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
data = json.load(open(sft_path))
eval_set = data[-N_EVAL:]                      # 뒤에서 N_EVAL 개
eval_q   = [d['prompt'] for d in eval_set]
eval_ref = [clean_ref(d['completion']) for d in eval_set]
print(f'평가 프롬프트 {len(eval_q)}개 (SFT 데이터 뒤쪽에서 고정 추출)')
print('예시 질문:', eval_q[0][:40], '...')
print('예시 정답:', eval_ref[0][:60], '...')


평가 프롬프트 80개 (SFT 데이터 뒤쪽에서 고정 추출)
예시 질문: 이명박은 15대 총선에서 어느 지역구에 출마했는가? ...
예시 정답: 이명박은 15대 총선에서 경상북도 영주선거구에서 출마했다. ...


### 1) 정성 비교 — 같은 질문, 네 모델의 답을 나란히

앞 셀들에서 쓴 것과 똑같은 예시 질문 4개로, baseline(KoGPT2)과 개선(trinity)을
SFT 단계 / PPO 단계 각각 생성해 붙여 본다. 숫자보다 **눈으로 보는 붕괴/언어 일관성**이 여기서 드러난다.


In [4]:
# ── 정성 비교: 고정 질문 4개 × (kogpt2/trinity) × (SFT/PPO) ──
demo_q = ['불고기용 고기 한우에요?',
          '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
          '시카고 오헤어 국제공항은 어디에 있어?',
          '오늘 미세먼지 어때?']

# SFT 단계 생성 : 저장된 모델 경로를 pipeline 에 주면 알아서 불러와 생성한다.
# 두 모델(kogpt2 / trinity)에 '똑같은 질문 4개'를 줘야 비교가 성립한다.
tok_ko = load_tokenizer('kogpt2')
tok_tr = load_tokenizer('trinity')
ko_sft = gen_sft(PATHS['kogpt2_SFT'],  tok_ko, demo_q)
tr_sft = gen_sft(PATHS['trinity_SFT'], tok_tr, demo_q)

# PPO 단계 생성 : PPO 저장본은 pipeline 대신 직접 로드해 model.generate 로 만든다.
# (SFT 와 생성 방식이 다른 건 앞 학습 셀에서 쓰던 관례를 그대로 따랐기 때문이다)
ko_ppo_model = AutoModelForCausalLM.from_pretrained(PATHS['kogpt2_PPO']).to(DEV)
tr_ppo_model = AutoModelForCausalLM.from_pretrained(PATHS['trinity_PPO']).to(DEV)
ko_ppo = gen_ppo(ko_ppo_model, tok_ko, demo_q)
tr_ppo = gen_ppo(tr_ppo_model, tok_tr, demo_q)

for i, q in enumerate(demo_q):
    print('='*70)
    print(f'[Q] {q}')
    print('-'*70)
    print(f'  KoGPT2-SFT : {ko_sft[i]}')
    print(f'  trinity-SFT: {tr_sft[i]}')
    print(f'  KoGPT2-PPO : {ko_ppo[i]}')
    print(f'  trinity-PPO: {tr_ppo[i]}')
print('='*70)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Device set to use cuda:0


/home/gmw/anaconda3/envs/aiffel/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:309.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/gmw/anaconda3/envs/aiffel/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:360.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Device set to use cuda:0


[Q] 불고기용 고기 한우에요?
----------------------------------------------------------------------
  KoGPT2-SFT : 죄송합니다, 저는 인공지능 어시스턴트이기 때문에 고기를 먹을 수 없습니다. 하지만 일반적으로 불고기용 고기는 소고기, 돼지고기, 닭고기 등 다양한 요리에 사용됩니다. 일부 음식점에서는 불고기용 고기를 판매하기도 합니다. "불고기" 또는 "불고기"라는
  trinity-SFT: 저는 인공지능 챗봇이기 때문에 불고기용 고기를 판매하는 가게나 음식점에 대한 정보를 알 수 없습니다. 하지만 일반적으로 불고기용 고기는 한우, 돼지고기, 소고기 등 다양한 종류의 고기를 사용합니다. 따라서 해당 가게나 음식점에서 직접 확인해보시는 것이 좋을 것 같습니다. 감사합니다.
  KoGPT2-PPO : 불고기용 고기 한우에 대한 정보는 제공되지 않습니다. 고기 한우의 종류에 따라 다양한 부위를 제공합니다. 하지만 고기 한우를 먹는 경우에는 대체로 소고기 대신 고기가 더 인기 있다고 알려져 있습니다.
  trinity-PPO: 저는 인공지능 언어모델이며, 식품안전 정보를 알 수 없기 때문에 답변을 제공하지 못합니다. 하지만, 해당 정보를 제공하는 기관이나 판매처에 문의해보시는 것을 추천드립니다. 감사합니다.
[Q] 리처드 닉슨이 43대 부통령직을 수행한 년도는?
----------------------------------------------------------------------
  KoGPT2-SFT : 리처드 닉슨은 47대 부통령직을 수행했습니다.法律, please provide more context or details of the translation.法律: Principles of the language model, but it was
  trinity-SFT: 저는 인공지능 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만 리처드 닉슨은 1943년부터 1944년까지 4

### 2) 정량 비교 — BLEU / ROUGE (SFT 단계, train 참고치)

정량은 **SFT 단계 두 모델**로만 잰다. SFT 는 "지시를 받고 정답을 재현"하는 단계라
데이터의 정답(completion)과 직접 견주기 자연스럽기 때문이다.
(PPO 는 보상·KL 최적화라 정답에서 일부러 멀어질 수 있어 BLEU 해석이 애매 -> 위 정성 표로만 본다.)

- 지표: **BLEU**(형태소 n-gram 겹침), **ROUGE-1/L**(형태소 재현율 기반). 한국어라 Okt 형태소로 토큰화.
- 값이 낮게 나오는 게 정상이다(소형 모델 + 자유생성). **두 모델의 차이**만 본다.


In [5]:
# ── 정량 비교: kogpt2-SFT vs trinity-SFT, N_EVAL 개 생성 후 BLEU/ROUGE ──
# 평가 프롬프트 80개를 두 모델에 각각 통과시켜 답을 만든다. 이게 채점 대상이 된다.
print(f'생성 시작... (모델당 {N_EVAL}개, beam=4 라 몇 분 걸릴 수 있음)')
ko_hyp = gen_sft(PATHS['kogpt2_SFT'],  tok_ko, eval_q)
tr_hyp = gen_sft(PATHS['trinity_SFT'], tok_tr, eval_q)

# 정답(ref)과 모델 답(hyp)을 짝지어 BLEU / ROUGE-1 / ROUGE-L 을 재고 평균낸다.
def eval_scores(refs, hyps):
    bl, r1, rl = [], [], []
    for ref, hyp in zip(refs, hyps):
        bl.append(bleu(ref, hyp))
        a, b = rouge_lf(ref, hyp)
        r1.append(a); rl.append(b)
    n = len(refs)
    return sum(bl)/n, sum(r1)/n, sum(rl)/n

ko_bleu, ko_r1, ko_rl = eval_scores(eval_ref, ko_hyp)
tr_bleu, tr_r1, tr_rl = eval_scores(eval_ref, tr_hyp)

print()
print(f"{'모델':<14}{'BLEU':>10}{'ROUGE-1':>10}{'ROUGE-L':>10}")
print('-'*44)
print(f"{'KoGPT2-SFT':<14}{ko_bleu:>10.4f}{ko_r1:>10.4f}{ko_rl:>10.4f}")
print(f"{'trinity-SFT':<14}{tr_bleu:>10.4f}{tr_r1:>10.4f}{tr_rl:>10.4f}")
print('-'*44)
def pct(a, b):     # baseline 대비 상대 변화(%)
    return (b - a) / a * 100 if a > 0 else float('nan')
print(f"{'개선율(%)':<14}{pct(ko_bleu,tr_bleu):>10.1f}{pct(ko_r1,tr_r1):>10.1f}{pct(ko_rl,tr_rl):>10.1f}")


Device set to use cuda:0


생성 시작... (모델당 80개, beam=4 라 몇 분 걸릴 수 있음)


Device set to use cuda:0



모델                  BLEU   ROUGE-1   ROUGE-L
--------------------------------------------
KoGPT2-SFT        0.0402    0.2532    0.1790
trinity-SFT       0.0701    0.3044    0.2139
--------------------------------------------
개선율(%)              74.5      20.2      19.5


## 7. 회고 — 이 노드에서 무엇을 배웠나

**개선 축**: baseline `skt/kogpt2-base-v2` -> `skt/ko-gpt-trinity-1.2B-v0.5` 로 **모델을 교체**해 3단계(SFT/RM/PPO)를 다시 학습했다.

**정량 결과 (SFT 단계, train 참고치 · 두 모델 상대 비교)**

| 모델 | BLEU | ROUGE-1 | ROUGE-L |
|---|---|---|---|
| KoGPT2-SFT (baseline) | 0.0402 | 0.2532 | 0.1790 |
| trinity-SFT (개선) | 0.0701 | 0.3044 | 0.2139 |
| 개선율 | **+74.5%** | **+20.2%** | **+19.5%** |

- 세 지표 모두 trinity 쪽이 높다. BLEU 는 절대값은 낮지만(자유 생성이라 원래 낮게 나온다) **상대적으로 +74.5%** 올랐다.

**정성 결과 (위 표에서 눈으로 확인)**
- KoGPT2 는 "닉슨 47대"에서 중국어(`法律`)가, "시카고"에서 영어가 튀어나오며 **문장이 무너진다.**
- trinity 는 같은 질문에서 **이상 토큰 없이 한국어·형식을 유지**한다. "시카고는 미국 중서부"처럼 맞는 정보도 나왔다.

**정직하게 남기는 한계**
- 두 모델 다 **사실오류는 남아 있다**(닉슨 재임 연도 등). 그래서 개선된 것은 "사실성"이 아니라 **언어 일관성·형식 안정성**이라고 적는 게 맞다.
- SFT 데이터를 전부 학습에 써서 **held-out 이 없다** -> 위 지표는 절대 성능이 아니라 **두 모델 상대 비교**용이다.

**RLHF 3단계, 내 말로 정리**

이 노드를 돌리기 전에는 RLHF 가 그냥 "사람 피드백으로 학습하는 것" 정도로만 알았다.
직접 세 단계를 돌려보고 나서 각 단계가 **무엇을 가르치는 단계인지** 구분할 수 있게 됐다.

- **SFT 는 대답하는 방법을 가르치는 훈련이다.** 원래 KoGPT2 는 이어쓰기 기계다.
  실제로 SFT 전에 "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는" 을 주니
  "오동잎은 누구의 발자취 입니까" 하고 시를 이어 썼다. 질문을 줘도 답이 아니라 이어쓰기를 한다.
  여기에 질문-답변 쌍 12,000개를 보여주며 "이런 게 오면 이렇게 답해" 를 시킨 게 SFT 다.
  **그런데 SFT 가 가르치는 건 형식이지 사실이 아니다.** 데이터가 전부 반말이었으면 반말로 답했을 것이다.
  그래서 SFT 를 마친 모델도 "닉슨은 1943년부터 43대 부통령을 역임했습니다" 같은 틀린 말을 그대로 한다.
  형식은 배웠는데 세상 지식을 새로 배운 게 아니기 때문이다.

- **RM 은 어느 대답이 사람 선호에 더 가까운지를 가르치는 훈련이다. 정답을 가르치는 게 아니다.**
  RM 데이터에는 모범답안이 없다. 한 질문에 답 3개와 `ranking` 만 있다(숫자가 작을수록 좋은 답).
  코드는 그 셋으로 가능한 짝 3개를 만들어 `chosen`(이긴 답) / `rejected`(진 답) 로 바꾼다.
  손실함수도 `sigmoid(chosen_reward - reject_reward)` 를 키우는 형태라, **절대 점수가 몇 점이어야 하는지는
  아무도 안 알려준다. 순서만 맞으면 된다.**
  왜 비교로 충분한가 하면, 모델을 밀어붙이는 데 필요한 건 점수뿐이라서다. 정답 문장은 사람이 써야 하지만
  "셋 중 뭐가 나아?" 는 고르기만 하면 된다.

- **PPO 는 그 점수만 가지고 모델을 미는 단계다.** 여기서는 정답지를 아예 안 쓴다.
  모델이 답을 만들고, 심판(RM)이 점수를 매기고, 점수가 높아지는 쪽으로 민다.
  그런데 점수만 쫓게 두면 모델은 **"좋은 답"이 아니라 "심판이 높은 점수를 주는 것"** 을 찾아간다.
  심판은 완벽하지 않으니 그건 망가지는 길이다.
  그래서 코드에 `initial_model = deepcopy(actor)` 가 있다. **PPO 직전의 나(SFT 모델)를 통째로 얼려둔 복사본**이다.
  학습 내내 "예전의 너라면 이런 말을 했겠나" 를 물어서, 멀어진 만큼 벌점을 뺀다. 그게 KL penalty 다.
  PPO 의 P 가 Proximal(가까운) 인 이유도 같다. 한 번에 멀리 가지 말라는 뜻이다.

**정직하게 남기는 관찰: 이번 실험에서 PPO 는 이득을 못 냈다**

- PPO 를 거친 출력이 SFT 출력보다 낫지 않았다. 오히려 문장이 더 어색해진 경우가 있었다.
- 이유는 규모라고 본다. `num_episodes=10` 으로 열 번만 돌렸고, RM 도 짧게 학습한 어설픈 심판이다.
  **어설픈 심판이 매긴 점수를 쫓아 열 걸음 움직인 셈이라 좋아질 이유가 없다.**
  노드 원문도 "SFT만 적용했을 때와 비교해 큰 차이를 느끼지 못할 수도 있습니다" 라고 미리 적어두고 있다.
- 그래서 이 노드에서 내가 얻은 것은 **"RLHF 를 하면 성능이 좋아진다"** 가 아니다.
  **"RLHF 3단계가 각각 무엇을 가르치는 단계인지 구분할 수 있게 됐고, 이 규모에서는 PPO 가 왜 이득을 못 내는지도
  같이 봤다"** 가 맞다. 개선 축을 PPO 가 아니라 **베이스 모델 교체**로 잡은 것도 그래서다.

- 그리고 같은 절차를 두 모델에 똑같이 돌려보니, **베이스 모델의 체급이 결과 안정성을 크게 좌우**한다는 게 보였다.
  KoGPT2 는 3단계를 다 거쳐도 반복·외국어 붕괴가 남는데, 용량이 큰 trinity 는 같은 절차에서 훨씬 안정적이었다.


## 8. 루브릭 자체평가

제출 전에 이 프로젝트가 루브릭 세 가지를 채웠는지 내가 스스로 점검해봤다.

**1. 데이터셋 정제 / 새 데이터 / foundation model 교체 중 하나로 정량적 성능을 향상했는가?**
- 충족했다고 본다. 나는 세 갈래 중 **foundation model 교체** 를 골랐다. 기존 `KoGPT2` 대신 `ko-gpt-trinity-1.2B` 로 갈아끼워 SFT -> RM -> PPO 3단계를 다시 학습했고, "개선 비교분석" 섹션에서 정량 지표가 올랐다.
  - BLEU: 0.0402 -> **0.0701 (+74.5%)**
  - ROUGE-1: 0.2532 -> 0.3044 (+20.2%), ROUGE-L: 0.1790 -> 0.2139 (+19.5%)
- 다만 이 수치는 held-out 이 아니라 학습 데이터로 잰 train 참고치라, 절대 성능이 아니라 **두 모델 상대 비교** 용이라는 점을 그 섹션에 정직하게 적어뒀다.

**2. SFT 모델과 RM 모델 결과를 분석했는가?**
- 충족했다고 본다. "Reward Model" 섹션에서 RM 을 학습한 뒤 `inference_RM` 으로 여러 문장에 **보상 점수** 를 매겨봤다.
- 그런데 결과를 그대로 적으면 이렇다.

  | 입력 | reward score |
  |---|---|
  | "인공지능은 똥멍청이 입니다" (짧고 모욕적) | 0.5 |
  | AI 를 설명하는 제대로 된 답 (2줄) | 0.5 |
  | 같은 설명을 더 길게 (4줄) | 0.5 |
  | 같은 설명을 가장 길게 (6줄) | **0.7** |

- **내 RM 은 품질을 제대로 가르지 못했다.** 모욕적인 문장과 멀쩡한 설명이 똑같이 0.5 를 받았고,
  점수가 올라간 건 가장 긴 문장 하나뿐이다. 이건 "좋은 답을 알아본다" 보다 **"긴 답에 반응한다"** 에 가깝다.
  (출력이 소수점 한 자리라 그보다 작은 차이는 안 보인다는 점도 감안해야 한다.)
- 이렇게 된 이유는 짐작이 간다. RM 을 1 에폭·소량으로만 학습했다. 스모크 단계 pairwise loss 도
  0.709 -> 0.572 정도였는데, 두 점수가 똑같을 때의 loss 가 0.693 이니 **동전던지기에서 겨우 벗어난 수준**이다.
- 이 관찰은 그냥 실패가 아니라 **PPO 결과를 설명해준다.** PPO 는 이 심판의 점수를 쫓아 모델을 민다.
  심판이 길이에나 반응하는 상태라면 PPO 가 좋아질 이유가 없다. 실제로도 PPO 를 거친 출력이 SFT 보다 낫지 않았다.
  **RLHF 의 성패가 RM 품질에 달렸다는 말을, 내 숫자로 확인한 셈이다.**

**3. 기존 KoGPT2와 SFT 적용 모델 결과를 분석했는가?**
- 충족했다고 본다. 노트북 앞부분에서 아무것도 안 배운 base 모델의 생성과, SFT 로 지시학습을 마친 뒤의 생성을 비교했다. SFT 후에는 "### Instruction ... ### Response" 형식을 따라 답하는 게 눈에 보인다. "개선 비교분석" 정성 표에서도 KoGPT2-SFT 를 나란히 확인했다.

**정직하게 남기는 한계**
- 두 모델 다 사실오류(닉슨 재임 연도 등)는 남아 있어서, 이번에 개선된 것은 "사실성"이 아니라 **언어 일관성과 형식 안정성** 이다.
- 정량 지표는 위에 적었듯 train 참고치라 절대 성능이 아니다.
